# Per-Stock Training

Trains one LSTM per stock on the 49 tickers with pre-computed sentiment.
Matches the reference paper's methodology:

- **Window**: 20 trading days, sentiment-gated (anchor day must have news)
- **Architecture**: LSTM(input=32, hidden=32, layers=2) with sentiment projected 768→16
- **Hyperparameters**: lr=1e-3, StepLR(10, 0.1), 150 epochs, batch=16, no early stopping
- **Split**: train < 2023-04, val = last 10% of pre-June-2023, test ≥ 2023-06

Prerequisites:
- `data/prices/data/historical-prices/<SYMBOL>/<YEAR>.csv`
- `data/sentiment/data/sentiment/<SYMBOL>.parquet`

In [1]:
from __future__ import annotations

import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import src
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("train_per_stock")

## Config

In [2]:
from src.training import ComputeConfig, TrainingConfig

CUTOFF      = "2023-06-01"
VAL_FRAC    = 0.1
PRICE_YEARS = list(range(2018, 2025))
SEED        = 42

config = TrainingConfig.reference(seed=SEED)  # lr=1e-3, StepLR, 150 epochs, window=20, batch=16
compute_config = ComputeConfig(num_workers=0)
compute_config.setup()

print(f"Device : {compute_config.device}")
print(f"Window : {config.window}")
print(f"LR     : {config.lr}")
print(f"Epochs : {config.n_epochs}")
print(f"Sched  : {config.scheduler} (step={config.step_size}, γ={config.gamma})")

Device : cpu
Window : 20
LR     : 0.001
Epochs : 150
Sched  : step (step=10, γ=0.1)


## Discover tickers with sentiment data

In [3]:
from src.repositories.prices import PriceRepository
from src.repositories.sentiment import SentimentRepository

# Actual data paths (nested inside data/ due to DVC structure)
PRICE_DIR = Path("../data/historical-prices/prices/data/historical-prices")
SENT_DIR  = Path("../data/sentiment/data/sentiment")

price_repo = PriceRepository(data_dir=PRICE_DIR)
sent_repo  = SentimentRepository(data_dir=SENT_DIR)

# Find tickers that have both price and sentiment data
sent_tickers = sorted(t for t in (
    f.stem for f in SENT_DIR.glob("*.parquet")
))
tickers = []
for t in sent_tickers:
    try:
        price_repo.load(t, 2024)
        tickers.append(t)
    except FileNotFoundError:
        print(f"  Skipping {t}: no price data")

print(f"\nTickers with both prices and sentiment: {len(tickers)}")
print(tickers)

  Skipping TSM: no price data

Tickers with both prices and sentiment: 49
['AAPL', 'ABBV', 'ADBE', 'AMD', 'AMT', 'AMZN', 'AVGO', 'BA', 'BAC', 'CAT', 'COP', 'COST', 'CVX', 'DIS', 'DUK', 'GE', 'GOOGL', 'GS', 'HD', 'HON', 'INTC', 'JNJ', 'JPM', 'KO', 'LIN', 'LLY', 'LMT', 'MA', 'MCD', 'META', 'MRK', 'MS', 'MSFT', 'NEE', 'NFLX', 'NKE', 'NVDA', 'ORCL', 'PFE', 'PG', 'SBUX', 'SLB', 'T', 'TSLA', 'UNH', 'V', 'VZ', 'WMT', 'XOM']


## Train one model per stock

For each ticker:
1. Load prices + sentiment
2. Build `StockDataset` (computes tech factors, aligns sentiment)
3. Build sentiment-gated DataLoaders (only windows where anchor day has news)
4. Create fresh LSTM (32 hidden, 768→16 sentiment projection)
5. Train 150 epochs with reference hyperparameters
6. Evaluate on test set (post-cutoff) with bootstrap CIs
7. Save checkpoint + test predictions

In [4]:
from src.features.dataset import StockDataset, build_per_stock_loaders
from src.model.lstm import SentimentLSTM
from src.model.trainer import Trainer
from src.repositories.models import ModelRepository

model_repo = ModelRepository()

results: list[dict] = []
failed: list[str] = []

for i, ticker in enumerate(tickers):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(tickers)}] {ticker}")
    print(f"{'='*60}")

    # --- Load data ---
    try:
        price_df = price_repo.load_years(ticker, PRICE_YEARS)
    except FileNotFoundError:
        print(f"  No price data, skipping")
        failed.append(ticker)
        continue

    sentiment_df = sent_repo.load(ticker)

    # --- Build dataset ---
    try:
        ds = StockDataset(
            symbol=ticker,
            price_df=price_df,
            sentiment_df=sentiment_df,
            window=config.window,
        )
    except RuntimeError as exc:
        print(f"  Dataset error: {exc}")
        failed.append(ticker)
        continue

    # --- Build sentiment-gated loaders ---
    train_loader, val_loader, test_loader = build_per_stock_loaders(
        ds,
        cutoff=CUTOFF,
        val_frac=VAL_FRAC,
        batch_size=config.batch_size,
    )

    n_train = len(train_loader.dataset)
    n_val   = len(val_loader.dataset)
    n_test  = len(test_loader.dataset)

    if n_train == 0:
        print(f"  No training data, skipping")
        failed.append(ticker)
        continue

    print(f"  Windows — train: {n_train}, val: {n_val}, test: {n_test}")

    # --- Create model ---
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
    )

    # --- Train ---
    trainer = Trainer(model, config, compute_config)
    train_result = trainer.fit(train_loader, val_loader)

    print(
        f"  Best epoch: {train_result.best_epoch} | "
        f"val_loss: {train_result.best_val_loss:.4f} | "
        f"val_auc: {train_result.best_val_auc:.4f}"
    )

    # --- Evaluate test set ---
    if n_test > 0:
        eval_result = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)
        print(
            f"  Test AUC:  {eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )
        print(
            f"  Test Acc:  {eval_result.accuracy_mean:.3f} "
            f"[{eval_result.accuracy_ci_low:.3f}, {eval_result.accuracy_ci_high:.3f}]"
        )
    else:
        eval_result = None
        print("  No test data")

    # --- Save checkpoint ---
    ckpt_name = f"per_stock_lstm_{ticker}"
    model_repo.save(
        ckpt_name,
        model,
        {
            "ticker": ticker,
            "window": config.window,
            "n_train": n_train,
            "n_val": n_val,
            "n_test": n_test,
            "best_epoch": train_result.best_epoch,
            "best_val_loss": train_result.best_val_loss,
            "best_val_auc": train_result.best_val_auc,
            "test_auc": eval_result.auc_mean if eval_result else None,
            "test_accuracy": eval_result.accuracy_mean if eval_result else None,
            "history": train_result.history,
        },
    )

    results.append({
        "ticker": ticker,
        "n_train": n_train,
        "n_test": n_test,
        "best_epoch": train_result.best_epoch,
        "val_loss": train_result.best_val_loss,
        "val_auc": train_result.best_val_auc,
        "test_auc": eval_result.auc_mean if eval_result else None,
        "test_acc": eval_result.accuracy_mean if eval_result else None,
    })

print(f"\n\nDone. Trained: {len(results)}, Failed: {len(failed)}")
if failed:
    print(f"Failed tickers: {failed}")

13:44:40 INFO     src.features.dataset  AAPL: 1646 windows, 16 tech features
13:44:40 INFO     src.features.dataset  AAPL — sentiment-gated: train=1053, val=117, test=383 (of 1646 total windows)



[1/49] AAPL
  Windows — train: 1053, val: 117, test: 383


13:44:42 INFO     src.model.trainer  Epoch   1 | train_loss=0.7415 | val_loss=0.6893 | val_auc=0.3865 | val_acc=0.5897
13:44:43 INFO     src.model.trainer  Epoch   2 | train_loss=0.7182 | val_loss=0.6823 | val_auc=0.4885 | val_acc=0.5726
13:44:44 INFO     src.model.trainer  Epoch   3 | train_loss=0.7032 | val_loss=0.8383 | val_auc=0.4952 | val_acc=0.5812
13:44:44 INFO     src.model.trainer  Epoch   4 | train_loss=0.7004 | val_loss=0.6690 | val_auc=0.5900 | val_acc=0.5897
13:44:45 INFO     src.model.trainer  Epoch   5 | train_loss=0.6910 | val_loss=0.6779 | val_auc=0.4970 | val_acc=0.5897
13:44:45 INFO     src.model.trainer  Epoch   6 | train_loss=0.6822 | val_loss=0.6778 | val_auc=0.4786 | val_acc=0.5812
13:44:46 INFO     src.model.trainer  Epoch   7 | train_loss=0.6776 | val_loss=0.6892 | val_auc=0.5033 | val_acc=0.5812
13:44:47 INFO     src.model.trainer  Epoch   8 | train_loss=0.6717 | val_loss=0.6771 | val_auc=0.4822 | val_acc=0.5983
13:44:47 INFO     src.model.trainer  Epoch   9 |

  Best epoch: 4 | val_loss: 0.6690 | val_auc: 0.5900


13:45:50 INFO     src.features.dataset  ABBV: 1646 windows, 16 tech features
13:45:50 INFO     src.features.dataset  ABBV — sentiment-gated: train=494, val=54, test=249 (of 1646 total windows)


...

  Best epoch: 9 | val_loss: 0.6517 | val_auc: 0.7570


14:21:59 INFO     src.features.dataset  XOM: 1646 windows, 16 tech features
14:21:59 INFO     src.features.dataset  XOM — sentiment-gated: train=522, val=58, test=285 (of 1646 total windows)


  Test AUC:  0.436 [0.375, 0.493]
  Test Acc:  0.425 [0.373, 0.484]

[49/49] XOM
  Windows — train: 522, val: 58, test: 285


14:21:59 INFO     src.model.trainer  Epoch   1 | train_loss=0.8921 | val_loss=0.6886 | val_auc=0.5591 | val_acc=0.6207
14:21:59 INFO     src.model.trainer  Epoch   2 | train_loss=0.8112 | val_loss=0.7068 | val_auc=0.5663 | val_acc=0.4483
14:21:59 INFO     src.model.trainer  Epoch   3 | train_loss=0.7921 | val_loss=0.6997 | val_auc=0.6595 | val_acc=0.5000
14:21:59 INFO     src.model.trainer  Epoch   4 | train_loss=0.8102 | val_loss=0.7186 | val_auc=0.5448 | val_acc=0.4828
14:22:00 INFO     src.model.trainer  Epoch   5 | train_loss=0.7879 | val_loss=0.6695 | val_auc=0.6858 | val_acc=0.6552
14:22:00 INFO     src.model.trainer  Epoch   6 | train_loss=0.7494 | val_loss=0.6909 | val_auc=0.5173 | val_acc=0.5345
14:22:00 INFO     src.model.trainer  Epoch   7 | train_loss=0.7475 | val_loss=0.7513 | val_auc=0.5699 | val_acc=0.5172
14:22:00 INFO     src.model.trainer  Epoch   8 | train_loss=0.7227 | val_loss=0.6644 | val_auc=0.6535 | val_acc=0.5862
14:22:00 INFO     src.model.trainer  Epoch   9 |

  Best epoch: 8 | val_loss: 0.6644 | val_auc: 0.6535
  Test AUC:  0.560 [0.489, 0.625]
  Test Acc:  0.551 [0.488, 0.607]


Done. Trained: 49, Failed: 0


## Results summary

In [5]:
df_results = pd.DataFrame(results).sort_values("test_auc", ascending=False)
print(df_results.to_string(index=False, float_format="%.3f"))
print(f"\nMean test AUC: {df_results['test_auc'].mean():.3f}")
print(f"Mean test Acc: {df_results['test_acc'].mean():.3f}")
print(f"Stocks with AUC > 0.5: {(df_results['test_auc'] > 0.5).sum()} / {len(df_results)}")

ticker  n_train  n_test  best_epoch  val_loss  val_auc  test_auc  test_acc
     T      610     214           1     0.704    0.582     0.634     0.463
 GOOGL      963     383           2     0.685    0.446     0.586     0.581
    GS      612     302           2     0.677    0.646     0.580     0.534
    KO      383     153          17     0.646    0.709     0.576     0.548
   XOM      522     285           8     0.664    0.654     0.560     0.551
  ORCL      429     274           7     0.664    0.687     0.553     0.540
   CVX      435     244           3     0.683    0.559     0.543     0.517
   SLB      257     157           8     0.548    0.646     0.543     0.580
  AAPL     1053     383           4     0.669    0.590     0.539     0.600
   UNH      376     261          15     0.660    0.659     0.539     0.504
  TSLA     1009     383          19     0.670    0.632     0.537     0.516
  NFLX      886     350           5     0.671    0.569     0.536     0.521
  INTC      729     358  